In [ ]:
import geopandas as gpd
import xarray as xr
import rasterio as rio
import glob

In [ ]:
# Cut larger topo to size using input polygon
topo_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/snow17_topos/wg_blue_topo.nc'
poly_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts/blue_setup/polys/blue_river_basin_outline_HUC10dissolved_32613.shp'

In [ ]:
topo = xr.open_dataset(topo_fn)
topo

In [ ]:
# Write a crs to topo
topo.rio.write_crs('epsg:32613', inplace=True)
topo

In [ ]:
# Read in the topo.nc file on disk
topo_disk_fn = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/isnobal_scripts/blue_setup/output_100m/topo.nc'
topo_disk = xr.open_dataset(topo_disk_fn)
topo_disk.rio.write_crs('epsg:32613', inplace=True)
topo_disk

In [ ]:
# Reproject and match the topo to the same crs as the disk topo
topo_clip = topo.rio.reproject_match(topo_disk)
topo_clip

In [ ]:
topo_clip['cbrfc_zone'].plot()

# Now clip to the polygon

In [ ]:
poly = gpd.read_file(poly_fn)
poly

In [ ]:
poly.crs

In [ ]:
# Use polygon to clip the topo
topo_clip_poly = topo_clip.rio.clip(poly.geometry, poly.crs)#, drop=True)
# Make sure the topo_clip_poly is the same size as the topo_disk
topo_clip_poly = topo_clip_poly.rio.reproject_match(topo_disk, align=True)
topo_clip_poly

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
fig, axa = plt.subplots(1, 2, figsize=(12, 4))
topo_clip_poly['cbrfc_zone'].plot(ax=axa[0])
topo_clip['cbrfc_zone'].plot(ax=axa[1])
for ax in axa.flatten():
    ax.set_aspect('equal')
    ax.set_title('')

In [ ]:
# Set no data value
topo_clip_poly['cbrfc_zone'].attrs['nodata'] = 0

In [ ]:
# Reassign no data values to this values
topo_clip_poly['cbrfc_zone'] = topo_clip_poly['cbrfc_zone'].where(topo_clip_poly['cbrfc_zone'] > 0, 0)
# Remove the _FillValue attribute
topo_clip_poly['cbrfc_zone'].attrs.pop('_FillValue', None)
topo_clip_poly['cbrfc_zone']

In [ ]:
# Mask the no data value areas when plotting
topo_clip_poly['cbrfc_zone'].plot()

In [ ]:
# Save to netcdf file
topo_clip_poly.to_netcdf('/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/ancillary/snow17_topos/blue_topo.nc')